In [5]:
import pandas as pd
import numpy as np

# 1. Carga de los archivos
df_movies = pd.read_csv('../data/netflix_movies_detailed_up_to_2025.csv')
df_shows = pd.read_csv('../data/netflix_tv_shows_detailed_up_to_2025.csv')

# Unificamos películas y series
df_shows['budget'] = np.nan
df_shows['revenue'] = np.nan
df = pd.concat([df_movies, df_shows], ignore_index=True)

# 2. LIMPIEZA CRÍTICA: Eliminar saltos de línea
df = df.replace(r'\n|\r', ' ', regex=True)

# Rellenar nulos
cols_to_fill = ['director', 'country', 'genres', 'language', 'rating']
df[cols_to_fill] = df[cols_to_fill].fillna('No Informado')

# 3. Preparación de variables para los KPIs
df['genero_principal'] = df['genres'].apply(lambda x: x.split(',')[0].strip() if isinstance(x, str) else x)
df['pais_principal'] = df['country'].apply(lambda x: x.split(',')[0].strip() if isinstance(x, str) else x)
df['idioma_principal'] = df['language'].apply(lambda x: x.split(',')[0].strip() if isinstance(x, str) else x)

# CORRECCIÓN DE FECHAS: Le decimos a pandas que lea el formato automático (YYYY-MM-DD)
df['date_added'] = pd.to_datetime(df['date_added'].str.strip(), errors='coerce')
df['año_agregado'] = df['date_added'].dt.year

# 4. Análisis Financiero y ROI
df['presupuesto'] = pd.to_numeric(df['budget'], errors='coerce')
df['ingresos'] = pd.to_numeric(df['revenue'], errors='coerce')
df['aplica_analisis_financiero'] = (df['presupuesto'] > 0) & (df['ingresos'] > 0)
df['roi'] = np.where(df['aplica_analisis_financiero'], 
                     (df['ingresos'] - df['presupuesto']) / df['presupuesto'], 
                     np.nan)

# 5. Exportar
ruta_exportacion = '../data/streamview_master_dataset.csv'
df.to_csv(ruta_exportacion, index=False, encoding='utf-8-sig')
print("Datos procesados y limpios para Power BI.")

Datos procesados y limpios para Power BI.
